# NeuralMath — Van wiskunde naar Python

We implementeren nu de voorwaartse propagatie, de kettingregel en de hierboven afgeleide stochastische updates met uitsluitend NumPy [ 4 ] . Om een niet-lineaire grens te tonen, breiden we de verborgen laag uit van drie naar twaalf ReLU-neuronen. Het resulterende \(2\) – \(12\) – \(1\) -netwerk heeft \(12(2+1)+(12+1)=49\) parameters en gebruikt dezelfde vergelijkingen.

**NumPy · 2–12–1 · 64 + 8 · 6000 · η = 0.05**

Constructie van de synthetische gegevens


## 01 — Voorbereiding van de omgeving


In [ ]:
"""Didactisch voorbeeld: fruitclassificatie met een 2-12-1-netwerk.
Afhankelijkheden: numpy en matplotlib. Uitvoering: python neural_network.py.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D




## 02 — ReLU-activering


In [ ]:
def relu(s):
    """Activatie van de verborgen laag: ReLU(s) = max(0,s)."""
    return np.maximum(0, s)




## 03 — Computationele afgeleide van ReLU


In [ ]:
def relu_derivative(s):
    # ReLU is niet differentieerbaar bij s=0; de berekende afgeleide wordt daar nul.
    # Dit is uitsluitend een conventie voor de terugpropagatiestap.
    return (s > 0).astype(float)




## 04 — Sigmoïde activering


In [ ]:
def sigmoid(s):
    """Uitvoeractivatie met waarden tussen 0 en 1."""
    return 1 / (1 + np.exp(-s))




## 05 — Hulpkromme voor gegevensgeneratie


In [ ]:
def reference_curve(m):
    # Kunstmatige kromme (cm): verschuiving, Gaussische piek, logistische stijging.
    # De constanten geven een niet-lineaire referentiekromme;
    # ze zijn geen metingen en geen geschatte parameters.
    return (6.68 + 0.74*np.exp(-((m-155)/20)**2)
            + 1.50/(1 + np.exp(-(m-202)/5)))




## 06 — Synthetische trainings- en illustratieve testgegevens


In [ ]:
def construct_data():
    # 16 massapunten in elk van vier stroken: a1, a2, l1, l2.
    masses_1 = np.linspace(118, 212, 16)
    # Kleine deterministische verschuivingen voorkomen identieke massa’s.
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7
    ])
    k = np.arange(16)

    # Appels: twee stroken boven de kromme, met kleine variaties in cm.
    a1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) + 0.22 + 0.05*np.sin(0.8*k)
    ])
    a2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) + 0.58 + 0.06*np.cos(0.45*k)
    ])

    # Sinaasappels: twee stroken onder de referentiekromme.
    l1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) - (0.22 + 0.04*np.cos(0.7*k))
    ])
    l2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) - (0.55 + 0.05*np.sin(0.5*k))
    ])

    X_train = np.vstack([a1, a2, l1, l2])
    y_train = np.array([1.0]*32 + [0.0]*32)

    # Acht illustratieve voorbeelden; geen uitgebreide statistische test.
    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45],
        [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86],
        [189.0, 6.48], [208.0, 7.55]
    ])
    y_test = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=float)
    return X_train, y_train, X_test, y_test




## 07 — Voorspellingen via voorwaartse propagatie


In [ ]:
def predict(X, mean, std, W1, b1, W2, b2):
    # x_j^* = (x_j - mu_j)/sigma_j, met statistieken uitsluitend uit de trainingsgegevens.
    Xn = (X - mean) / std
    # Voorwaartse stap in matrixnotatie (een rij per vrucht).
    h = relu(Xn @ W1.T + b1)
    return sigmoid(h @ W2 + b2)




## 08 — Training via terugpropagatie en gradiëntafdaling


In [ ]:
def train(Xn, y_train, eta=0.05, n_epochs=6000, seed=42):
    rng = np.random.default_rng(seed)
    # Kleine willekeurige gewichten doorbreken de symmetrie;
    # de biases beginnen op nul; schaal 0.1 is didactisch.
    W1 = 0.1 * rng.standard_normal((12, 2))
    b1 = np.zeros(12)
    W2 = 0.1 * rng.standard_normal(12)
    b2 = 0.0
    history = []

    for epoch in range(n_epochs):
        # SGD: update per vrucht, in elk tijdperk nieuw geschud.
        for i in rng.permutation(len(Xn)):
            x, target = Xn[i], y_train[i]

            # Voorwaarts: vergelijkingen van verborgen laag en uitvoerlaag.
            z1 = W1 @ x + b1
            h = relu(z1)
            z2 = W2 @ h + b2
            y_hat = sigmoid(z2)

            # Terugpropagatie: delta2 = dE/dz2 (kwadratisch verlies).
            delta2 = (y_hat - target) * y_hat * (1 - y_hat)
            dW2, db2 = delta2 * h, delta2
            # delta1_j = delta2 * W2_j * ReLU'(z1_j).
            delta1 = (delta2 * W2) * relu_derivative(z1)
            dW1 = np.outer(delta1, x)
            db1 = delta1

            # Stochastische afdaling: theta <- theta - eta*dE/dtheta.
            # Alle gradiënten gebruiken de OUDE parameterwaarden.
            W1 -= eta * dW1
            b1 -= eta * db1
            W2 -= eta * dW2
            b2 -= eta * db2

        # J: gemiddeld trainingsverlies na elk tijdperk.
        h_all = relu(Xn @ W1.T + b1)
        y_pred = sigmoid(h_all @ W2 + b2)
        history.append(np.mean(0.5 * (y_train - y_pred)**2))

    return W1, b1, W2, b2, np.array(history)




## 09 — Weergave van de geleerde grens


In [ ]:
def plot_results(X_train, y_train, X_test, y_test,
                  mean, std, W1, b1, W2, b2, save_path=None):
    # Niveau y_hat=0.5: GELEERDE beslissingsgrens.
    masses = np.linspace(116, 214, 500)
    diameters = np.linspace(6.0, 8.85, 500)
    M, D = np.meshgrid(masses, diameters)
    grid = np.column_stack([M.ravel(), D.ravel()])
    P = predict(grid, mean, std, W1, b1, W2, b2).reshape(M.shape)

    fig, ax = plt.subplots(figsize=(9.2, 6.3))
    ax.contourf(M, D, P, levels=[0, 0.25, 0.50, 0.75, 1],
                colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
                alpha=0.95)
    ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'],
               linewidths=2.2)

    mask = y_train == 1.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_train == 0.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_test == 1.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^',
               s=92, color='#1f77b4')
    mask = y_test == 0.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x',
               s=110, color='#ff7f0e', linewidths=2.0)

    ax.set_xlabel('Massa (g)')
    ax.set_ylabel('Diameter (cm)')
    ax.set_title('Netwerkuitvoer na training')
    handles = [
        Line2D([0], [0], marker='o', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Appel - training'),
        Line2D([0], [0], marker='s', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Sinaasappel - training'),
        Line2D([0], [0], marker='^', markersize=9, color='#1f77b4',
               linestyle='None', label='Appel - test'),
        Line2D([0], [0], marker='x', markersize=10, color='#ff7f0e',
               linestyle='None', label='Sinaasappel - test'),
        Line2D([0], [0], color='#5a1a8a', linewidth=2.2,
               label='Geleerde beslissingsgrens')
    ]
    ax.legend(handles=handles, loc='center left',
              bbox_to_anchor=(1.02, 0.5), frameon=True)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    return fig, ax




## 10 — Uitvoering van het experiment


In [ ]:
if __name__ == '__main__':
    X_train, y_train, X_test, y_test = construct_data()
    mean, std = X_train.mean(axis=0), X_train.std(axis=0)
    Xn = (X_train - mean) / std
    W1, b1, W2, b2, history = train(Xn, y_train)
    p_train = predict(X_train, mean, std, W1, b1, W2, b2)
    p_test = predict(X_test, mean, std, W1, b1, W2, b2)
    print('Gemiddeld eindverlies:', history[-1])
    print('Correcte trainingsvoorspellingen:', np.sum((p_train >= 0.5) == y_train), '/64')
    print('Correcte illustratieve testvoorspellingen:',
          np.sum((p_test >= 0.5) == y_test), '/8')
    plot_results(X_train, y_train, X_test, y_test,
                 mean, std, W1, b1, W2, b2)
    plt.show()
